In [4]:
!pip install datasets

import torch
import torch.nn as nn
import torch.optim as optim
import re
from collections import Counter
from datasets import load_dataset

In [5]:
dataset = load_dataset("imdb")

train_data = dataset['train']
test_data = dataset['test']

In [6]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    return text

In [7]:
def tokenize(text):
    return text.split()

In [8]:
processed_data = []

for item in train_data.select(range(5000)):  # small subset
    text = clean_text(item['text'])
    tokens = tokenize(text)
    label = item['label']
    processed_data.append((tokens, label))

In [9]:
def build_vocab(data, max_size=10000):
    counter = Counter()
    for tokens, _ in data:
        counter.update(tokens)

    vocab = {word: i+2 for i, (word, _) in enumerate(counter.most_common(max_size))}
    vocab['<PAD>'] = 0
    vocab['<UNK>'] = 1
    return vocab

vocab = build_vocab(processed_data)

In [10]:
def encode(tokens, vocab):
    return [vocab.get(word, vocab['<UNK>']) for word in tokens]

def pad_sequence(seq, max_len=100):
    if len(seq) < max_len:
        seq += [0] * (max_len - len(seq))
    else:
        seq = seq[:max_len]
    return seq

In [11]:
X = []
y = []

for tokens, label in processed_data:
    encoded = encode(tokens, vocab)
    padded = pad_sequence(encoded)
    X.append(padded)
    y.append(label)

X = torch.tensor(X)
y = torch.tensor(y)

In [12]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        out = self.fc(hidden[-1])
        return torch.sigmoid(out)

In [13]:
model = LSTMModel(len(vocab), 64, 128)

In [14]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [16]:
for epoch in range(3):
    total_loss = 0

    for i in range(len(X)):
        inputs = X[i].unsqueeze(0)
        label = y[i].float()

        optimizer.zero_grad()

        output = model(inputs).squeeze()
        loss = criterion(output, label)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 5)

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 16.3656
Epoch 2, Loss: 0.0056
Epoch 3, Loss: 0.0004


In [17]:
def predict(text):
    text = clean_text(text)
    tokens = tokenize(text)
    encoded = encode(tokens, vocab)
    padded = pad_sequence(encoded)

    tensor = torch.tensor(padded).unsqueeze(0)
    output = model(tensor)

    return "Positive" if output.item() > 0.5 else "Negative"

In [18]:
print(predict("Amazing movie!"))
print(predict("Worst film ever"))

Negative
Negative
